In [ ]:
import pandas as pd
import random
import csv
from tqdm import tqdm
import string
import json
import numpy as np

from collections import Counter

import spacy

nlp = spacy.load("nl_core_news_lg") 

import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ['hf_token_own']
)

In [ ]:
path_name = 'ChiSCor_master_df.csv'
df = pd.read_csv(path_name, index_col=0)

### Prepare POS counter

In [ ]:
from collections import defaultdict
pos_tags = []
pos_tag_story_begin = []
stories_with_er = defaultdict(list)
for story in df['story_raw']:
    doc = nlp(story)

    sent_list = list(doc.sents)

    for i, sent in enumerate(sent_list):
        first_word = sent[0]
        pos_tags.append(first_word.pos_)
        
pos_counter = Counter(pos_tag_story_begin)

In [ ]:
nouns = set()
adjectives = set()
verbs = set()

for story in df['story_raw']:
    for token in nlp(story):
        if token.pos_ == 'NOUN':
            nouns.add(str(token))
        elif token.pos_ == 'VERB':
            verbs.add(str(token))
        elif token.pos_ == 'ADJ':
            adjectives.add(str(token))

In [ ]:
nouns, adjectives, verbs = list(nouns), list(adjectives), list(verbs)

In [ ]:
def select_pos_tag(weighted=True):
    if weighted:
        tags = list(pos_counter.keys())
        frequencies = list(pos_counter.values())
        sample = random.choices(tags, weights=frequencies, k=1)[0]
    else:
        sample = pos_tags[random.randint(0, len(pos_tags) - 1)]
    
    return sample

In [ ]:
with open("narative_elements.json", "r", encoding="utf-8") as f:
    narrative_elements = json.load(f)

with open("verhaalthemas.json", "r", encoding="utf-8") as f:
    verhaalthemas = json.load(f)

### Example: generating with POS tag

In [ ]:
# def generate_user_prompt():
#     chosen_noun = random.choice(nouns)
#     chosen_adjective = random.choice(adjectives)
#     chosen_verb = random.choice(verbs)
#     chosen_pos_tag = select_pos_tag()
#     chosen_letter = random.choice(string.ascii_lowercase)
#     chosen_feature = random.choice(story_features)
#     element = random.choice(verhaalelementen)

#     prompt = f"""Vertel een verhaal. 
# Het verhaal moet het volgende werkwoord bevatten: {chosen_verb}, het volgende zelfstandig naamwoord: {chosen_noun} en het volgende bijvoegelijk naamwoord: {chosen_adjective}.
# Het verhaal moet het volgende kenmerk bevatten: {chosen_feature} en het volgende verhaal element: {element}.
# Begin het verhaal met een woord met het volgende pos-tag {chosen_pos_tag}."""

#     return prompt

# def generate_tweeked_user_prompt(*, narrative_elements=None):
#     random.seed(10)
#     element = random.choice(narrative_elements)
#     prompt = f"""Vertel een verhaal. Het verhaal moet het volgende narratieve element bevatten: {element}"""

#     return prompt

def generate_tweeked_user_prompt(chosen_pos_tag):
    prompt = f"""Vertel een verhaal. 
Begin het verhaal met een woord met het volgende pos-tag {chosen_pos_tag}."""

    return prompt

def generate_user_prompt():
    prompt = f"""Vertel een verhaal."""

    return prompt

In [ ]:
experiment = "llama"

if experiment == "llama":
    name = "llama3-8b"
    model="meta-llama/Llama-3.1-8B-Instruct:novita"

print(name)
#user_prompt = generate_user_prompt()

system_prompt = f"""
Je bent een verteller van een kort verhaal (rond de 200 woorden).
Je bent een kind tussen de 4 en 6 en je vertelt een verhaal aan klasgenoten. 
Je publiek bestaat uit kinderen van jouw leeftijd. 
Geef het verhaal geen titel of introductie.
"""

with open(f"baseline_research_prompting_{name}.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header only once if file is empty
    csvfile.seek(0, 2)  # move to end
    if csvfile.tell() == 0:  
        writer.writerow(["model", "system", "user", "completion1", "completion2", "completion3", "completion4", "completion5"])

    for pos in tqdm(pos_counter.keys()):
        completions = []
        user_prompt = generate_tweeked_user_prompt(pos)

        for _ in tqdm(range(5)):
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )

            completion = response.choices[0].message.content.strip()
            completions.append(completion)

        # Write each row immediately
        writer.writerow([response.model, system_prompt, user_prompt] + completions)


### Age experiment

In [ ]:
stories = []
models = []
ages_by_stories = []
n = 50

user_prompt = generate_user_prompt()

for age_set in [[4, 6], [6, 7], [7, 8], [8, 9], [9, 10], [10, 11], [11, 12]]:
    lower_age = age_set[0]
    upper_age = age_set[1]
    system_prompt = f"""
Je bent een verteller van een kort verhaal (rond de 200 woorden).
Je bent een kind tussen de {lower_age} en {upper_age} en je vertelt een verhaal aan klasgenoten. 
Je publiek bestaat uit kinderen van jouw leeftijd. 
Geef het verhaal geen titel of introductie.
"""

    for _ in tqdm(range(n)):
        response = client.chat.completions.create(
            model="google/gemma-3-27b-it:nebius",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        completion = response.choices[0].message.content.strip()
        models.append(response.model)
        stories.append(completion)
        ages_by_stories.append(f'{lower_age}_{upper_age}')

In [ ]:
ages_gemma = pd.DataFrame(
    {'model': models, 
    'age': ages_by_stories,
    'story': stories}
)

In [ ]:
ages_gemma.to_csv('prompting_results/age_research_ellaborateprompt_DD_gemma3-27b.csv')

## Baseline (not told by kids)

In [ ]:
experiment = "llama"

if experiment == "llama":
    name = "llama3-8b"
    model="meta-llama/Llama-3.1-8B-Instruct:novita"

system_prompt = f"""
Je bent een verteller van een kort verhaal (rond de 200 woorden).
Gebruik alleen hele simpele woorden die een 3-jarig kind kan begrijpen.
Geef het verhaal geen titel of introductie.
"""

user_prompt = generate_user_prompt()

with open(f"baseline_research_prompting_{name}.csv", "a", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    
    # Write header only once if file is empty
    csvfile.seek(0, 2)  # move to end
    if csvfile.tell() == 0:  
        writer.writerow(["model", "system", "user", "completion"])

    for _ in tqdm(range(600)):
        completions = []
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )

        completion = response.choices[0].message.content.strip()
        completions.append(completion)

        # Write each row immediately
        writer.writerow([response.model, system_prompt, user_prompt] + completions)
